# County-Level Alignment Between Presidential and Gubernatorial Voting in 2020

**DATASCI 151 Final Project**  
**Group Members / Student IDs:**

- Name 1 — ID
- Name 2 — ID
- Name 3 — ID
- Name 4 — ID

---

## Notebook Roadmap

This notebook is organized into the following sections:
1. Introduction
2. Data Description
3. Data Loading and Initial Inspection
4. Data Cleaning and Preparation
5. Merging and Construction of Main Analysis Table
6. Validation Checks
7. Descriptive Statistics
8. Results
9. Discussion

> This version is a framework notebook. It is intentionally structured with placeholders, comments, and code skeletons before the final analysis code is written.


## 1. Introduction

**Write 1–2 paragraphs here.**

Suggested content:
- Briefly explain the 2020 U.S. election context.
- State the project question clearly.
- Explain why comparing presidential and gubernatorial county results is interesting.
- Preview the structure of the notebook and the kinds of findings you will report.

**Draft project question:**

> Across the states in this dataset with gubernatorial races, how closely did county-level gubernatorial results track presidential results, and was greater divergence associated with higher presidential third-party vote share?


## 2. Data Description

**Write 1 paragraph here describing the tables you use.**

Tables currently planned:
- `president_county_candidate.csv`
- `president_county.csv`
- `governors_county_candidate.csv`
- `governors_county.csv`

Suggested content:
- What one row represents in each file
- How many observations are in each table
- What kinds of columns appear in each table
- Why these four tables are sufficient for the analysis


In [ ]:
# 3. Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)


In [ ]:
# 4. Load data

# Update paths if needed depending on where your notebook lives.
pres_cand = pd.read_csv('president_county_candidate.csv')
pres_total = pd.read_csv('president_county.csv')
gov_cand = pd.read_csv('governors_county_candidate.csv')
gov_total = pd.read_csv('governors_county.csv')

# Quick check
datasets = {
    'president_county_candidate': pres_cand,
    'president_county': pres_total,
    'governors_county_candidate': gov_cand,
    'governors_county': gov_total
}

for name, df in datasets.items():
    print(f'\n{name}')
    print('shape:', df.shape)
    print(df.head(3))


## 3. Data Loading and Initial Inspection

**Write a short paragraph here after inspecting the data.**

Suggested points to mention:
- Which states appear in both presidential and gubernatorial data
- Any immediate inconsistencies in naming or missing values
- Why you restrict the analysis to the overlap set of states


In [ ]:
# Identify overlapping states between presidential and gubernatorial county files

pres_states = set(pres_total['state'].dropna().unique())
gov_states = set(gov_total['state'].dropna().unique())
overlap_states = sorted(pres_states.intersection(gov_states))

print('Number of overlapping states:', len(overlap_states))
print(overlap_states)


In [ ]:
# Restrict all tables to overlapping states

pres_cand = pres_cand[pres_cand['state'].isin(overlap_states)].copy()
pres_total = pres_total[pres_total['state'].isin(overlap_states)].copy()
gov_cand = gov_cand[gov_cand['state'].isin(overlap_states)].copy()
gov_total = gov_total[gov_total['state'].isin(overlap_states)].copy()

print('Restricted presidential candidate rows:', pres_cand.shape)
print('Restricted governor candidate rows:', gov_cand.shape)


## 4. Data Cleaning and Preparation

**Write a paragraph here explaining your cleaning steps.**

Possible steps:
- Standardize county and state names if needed
- Check for duplicates
- Recode party labels into Democratic, Republican, and Other
- Handle missing candidate-party combinations
- Decide whether to use all-vote shares or two-party shares as the main metric


In [ ]:
# Example: inspect columns so you can decide on cleaning rules

for name, df in [('pres_cand', pres_cand), ('gov_cand', gov_cand), ('pres_total', pres_total), ('gov_total', gov_total)]:
    print(f'\n{name} columns:')
    print(df.columns.tolist())


In [ ]:
# TODO: standardize county/state strings here if necessary
# Example placeholders only — replace with real cleaning if needed.

def clean_location_columns(df):
    df = df.copy()
    df['state'] = df['state'].astype(str).str.strip()
    df['county'] = df['county'].astype(str).str.strip()
    return df

pres_cand = clean_location_columns(pres_cand)
pres_total = clean_location_columns(pres_total)
gov_cand = clean_location_columns(gov_cand)
gov_total = clean_location_columns(gov_total)


In [ ]:
# TODO: inspect raw party labels

print('President party labels:')
print(pres_cand['party'].value_counts(dropna=False).head(20))

print('\nGovernor party labels:')
print(gov_cand['party'].value_counts(dropna=False).head(20))


In [ ]:
# Recode parties into DEM / REP / OTHER

def recode_party(party_value):
    # TODO: refine this mapping after inspecting actual values in the file
    p = str(party_value).strip().upper()
    if 'DEM' in p:
        return 'DEM'
    elif 'REP' in p or 'REPUBLICAN' in p:
        return 'REP'
    else:
        return 'OTHER'

pres_cand['party_group'] = pres_cand['party'].apply(recode_party)
gov_cand['party_group'] = gov_cand['party'].apply(recode_party)

print(pres_cand['party_group'].value_counts())
print(gov_cand['party_group'].value_counts())


## 5. Merging and Construction of the Main Analysis Table

**Write one paragraph here describing your merge process.**

Suggested merge story:
1. Aggregate candidate-level votes to the county level separately for presidential and gubernatorial elections.
2. Merge those summaries with the county total-vote files.
3. Merge presidential and gubernatorial county summaries into one final analysis table.


In [ ]:
# Function requirement: summarize county-level results for a given office

def build_county_summary(candidate_df, office_name):
    """
    Returns one row per state-county with party vote totals, shares, winner, and margin.
    Replace column names below if the raw CSV uses different names.
    """
    df = candidate_df.copy()

    # TODO: replace 'total_votes' with the correct vote-count column if needed
    grouped = (
        df.groupby(['state', 'county', 'party_group'])['total_votes']
          .sum()
          .reset_index()
    )

    wide = grouped.pivot_table(
        index=['state', 'county'],
        columns='party_group',
        values='total_votes',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    for col in ['DEM', 'REP', 'OTHER']:
        if col not in wide.columns:
            wide[col] = 0

    wide[f'{office_name}_dem_votes'] = wide['DEM']
    wide[f'{office_name}_rep_votes'] = wide['REP']
    wide[f'{office_name}_other_votes'] = wide['OTHER']

    wide[f'{office_name}_two_party_total'] = wide[f'{office_name}_dem_votes'] + wide[f'{office_name}_rep_votes']
    wide[f'{office_name}_all_candidate_total'] = (
        wide[f'{office_name}_dem_votes'] +
        wide[f'{office_name}_rep_votes'] +
        wide[f'{office_name}_other_votes']
    )

    wide[f'{office_name}_dem_share_2party'] = wide[f'{office_name}_dem_votes'] / wide[f'{office_name}_two_party_total']
    wide[f'{office_name}_rep_share_2party'] = wide[f'{office_name}_rep_votes'] / wide[f'{office_name}_two_party_total']
    wide[f'{office_name}_other_share_all'] = wide[f'{office_name}_other_votes'] / wide[f'{office_name}_all_candidate_total']
    wide[f'{office_name}_dem_margin_2party'] = wide[f'{office_name}_dem_share_2party'] - wide[f'{office_name}_rep_share_2party']
    wide[f'{office_name}_winner'] = np.where(
        wide[f'{office_name}_dem_votes'] > wide[f'{office_name}_rep_votes'],
        'DEM',
        'REP'
    )

    keep_cols = [
        'state', 'county',
        f'{office_name}_dem_votes',
        f'{office_name}_rep_votes',
        f'{office_name}_other_votes',
        f'{office_name}_two_party_total',
        f'{office_name}_all_candidate_total',
        f'{office_name}_dem_share_2party',
        f'{office_name}_rep_share_2party',
        f'{office_name}_other_share_all',
        f'{office_name}_dem_margin_2party',
        f'{office_name}_winner'
    ]

    return wide[keep_cols]


In [ ]:
# Build county summaries for both offices

# TODO: confirm the vote column name before running build_county_summary.
# If the raw files do not use 'total_votes', change the function above.

pres_summary = build_county_summary(pres_cand, 'pres')
gov_summary = build_county_summary(gov_cand, 'gov')

print(pres_summary.head())
print(gov_summary.head())


In [ ]:
# TODO: merge in county total-vote files
# Replace the vote column names below once confirmed.

pres_total_small = pres_total[['state', 'county', 'total_votes']].copy()
pres_total_small = pres_total_small.rename(columns={'total_votes': 'pres_total_votes'})

gov_total_small = gov_total[['state', 'county', 'total_votes']].copy()
gov_total_small = gov_total_small.rename(columns={'total_votes': 'gov_total_votes'})

pres_summary = pres_summary.merge(pres_total_small, on=['state', 'county'], how='left')
gov_summary = gov_summary.merge(gov_total_small, on=['state', 'county'], how='left')


In [ ]:
# Main merged analysis table

analysis_df = pres_summary.merge(gov_summary, on=['state', 'county'], how='inner')

# Core engineered variables
analysis_df['winner_mismatch'] = analysis_df['pres_winner'] != analysis_df['gov_winner']
analysis_df['abs_dem_share_diff'] = (
    analysis_df['pres_dem_share_2party'] - analysis_df['gov_dem_share_2party']
).abs()
analysis_df['dem_overperformance'] = (
    analysis_df['gov_dem_share_2party'] - analysis_df['pres_dem_share_2party']
)
analysis_df['turnout_gap'] = analysis_df['pres_total_votes'] - analysis_df['gov_total_votes']

analysis_df.head()


## 6. Validation Checks

Use this section to confirm that your merged table is trustworthy.

Suggested checks:
- one row per state-county in the final dataset
- missing values in key columns
- whether candidate-level vote totals are close to county total votes
- whether the number of counties per state looks reasonable


In [ ]:
# Validation checks

print('Final analysis shape:', analysis_df.shape)
print('Duplicate state-county rows:', analysis_df.duplicated(['state', 'county']).sum())

key_cols = [
    'pres_dem_share_2party', 'gov_dem_share_2party',
    'pres_winner', 'gov_winner', 'winner_mismatch'
]

print('\nMissing values in key columns:')
print(analysis_df[key_cols].isna().sum())


## 7. Descriptive Statistics

**Write one paragraph here describing the main columns you will analyze.**

Main candidate variables to describe:
- presidential Democratic two-party share
- gubernatorial Democratic two-party share
- absolute difference between those shares
- presidential third-party share
- turnout gap


In [ ]:
# Descriptive statistics table

desc_cols = [
    'pres_dem_share_2party',
    'gov_dem_share_2party',
    'abs_dem_share_diff',
    'pres_other_share_all',
    'turnout_gap'
]

desc_table = analysis_df[desc_cols].describe().round(3)
desc_table


## 8. Results

Each subsection below should include:
- code
- a polished table or plot
- 2–4 sentences of markdown interpreting the result


### Result 1. How often did counties choose different parties for president and governor?


In [ ]:
# Overall mismatch rate

overall_mismatch = analysis_df['winner_mismatch'].mean()
print('Overall county-level mismatch rate:', round(overall_mismatch, 3))

# State-level mismatch table
mismatch_by_state = (
    analysis_df.groupby('state')['winner_mismatch']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'mismatch_rate', 'sum': 'num_mismatch', 'count': 'num_counties'})
    .sort_values('mismatch_rate', ascending=False)
    .round(3)
)

mismatch_by_state


In [ ]:
# Plot: mismatch rate by state

mismatch_by_state['mismatch_rate'].plot(kind='bar', figsize=(10, 5))
plt.title('County-Level Presidential/Gubernatorial Winner Mismatch Rate by State')
plt.xlabel('State')
plt.ylabel('Mismatch rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Interpretation notes:**
- Which states have the highest mismatch rates?
- Are split-ticket counties rare or fairly common?
- Does winner mismatch alone tell the full story?


### Result 2. How closely do presidential and gubernatorial Democratic vote shares line up?


In [ ]:
# Scatterplot of presidential vs governor Democratic two-party share

plt.figure(figsize=(7, 7))
plt.scatter(analysis_df['pres_dem_share_2party'], analysis_df['gov_dem_share_2party'], alpha=0.6)
plt.plot([0, 1], [0, 1])
plt.title('County Democratic Share: President vs Governor')
plt.xlabel('Presidential Democratic two-party share')
plt.ylabel('Gubernatorial Democratic two-party share')
plt.tight_layout()
plt.show()


In [ ]:
# State-level alignment summary

state_alignment = []

for state_name, subdf in analysis_df.groupby('state'):
    corr_val = subdf['pres_dem_share_2party'].corr(subdf['gov_dem_share_2party'])
    mad_val = subdf['abs_dem_share_diff'].mean()
    state_alignment.append({
        'state': state_name,
        'correlation': corr_val,
        'mean_abs_diff': mad_val,
        'n_counties': len(subdf)
    })

state_alignment = pd.DataFrame(state_alignment).sort_values('mean_abs_diff', ascending=False)
state_alignment.round(3)


**Interpretation notes:**
- Are counties generally close to the 45-degree line?
- Which states show the most divergence?
- Is correlation alone enough, or is mean difference more informative?


### Result 3. Is higher third-party presidential vote share associated with greater cross-office divergence?


In [ ]:
# Optional: bin presidential third-party vote share for a grouped comparison

analysis_df['pres_other_bin'] = pd.qcut(
    analysis_df['pres_other_share_all'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

third_party_summary = (
    analysis_df.groupby('pres_other_bin')['abs_dem_share_diff']
    .agg(['mean', 'median', 'count'])
    .round(3)
)

third_party_summary


In [ ]:
# Plot grouped divergence by third-party bin

third_party_summary['mean'].plot(kind='bar', figsize=(7, 5))
plt.title('Average Presidential/Gubernatorial Divergence by Presidential Third-Party Vote Share')
plt.xlabel('Presidential third-party vote-share group')
plt.ylabel('Mean absolute Democratic-share difference')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Interpretation notes:**
- Do counties with more third-party presidential voting also have more divergence?
- Is the relationship large or small?
- Is this association consistent enough to mention in the discussion?


### Result 4. Which counties are the largest outliers?


In [ ]:
# Top counties by divergence across offices

outlier_cols = [
    'state', 'county',
    'pres_dem_share_2party',
    'gov_dem_share_2party',
    'abs_dem_share_diff',
    'winner_mismatch'
]

largest_outliers = analysis_df[outlier_cols].sort_values('abs_dem_share_diff', ascending=False).head(10)
largest_outliers.round(3)


**Interpretation notes:**
- Do the biggest outliers involve winner changes, or just margin shifts?
- Are the outliers concentrated in a few states?
- Does this add something more concrete than statewide averages?


## 9. Discussion

**Write 1–2 paragraphs here.**

Suggested structure:
- Summarize the biggest empirical findings.
- Explain what they suggest about county-level cross-office voting alignment.
- Note limitations: only the states in the dataset with governor races, county-level returns only, no demographics, no causal claims.
- End with a short reflection on what the analysis contributes.


## 10. Final Checklist Before Submission

- [ ] All code runs top to bottom without errors
- [ ] Output is clean and not cluttered
- [ ] Tables are rounded and labeled
- [ ] Plots have titles and axis labels
- [ ] Every result section includes markdown interpretation
- [ ] Introduction and discussion are complete
- [ ] Notebook is exported as both `.ipynb` and `.html`
